# Kriging with the Vecchia log-likelihood — objective="VLL(m)" (Julia)

This notebook demonstrates the **Vecchia approximated log-likelihood** as a fit
objective. Vecchia (1988): log p(y) is approximated by the product of the
conditionals of each observation given its $m$ nearest previously-ordered
neighbors (greedy maxmin ordering, Guinness 2018):

$$\log L \approx \sum_i \log p(y_i \mid y_{N(i)}), \qquad |N(i)| \le m$$

Each evaluation costs $O(n\,m^3)$ instead of $O(n^3)$, is a valid Gaussian
density (sparse inverse Cholesky), and is exact for $m = n-1$. The usage could
not be simpler — it is just an `objective` string:

- `objective="VLL"` (default $m=30$) or `objective="VLL(m)"`.

After the fit, ONE exact factorization is performed at $\theta^*$, so
`predict`/`simulate`/`update` behave exactly as after an `"LL"` fit.

Steps:
1. Setup
2. Branin function
3. Design of experiments ($n = 1500$)
4. Fit with `"VLL(30)"` vs exact `"LL"`: wall time and hyperparameters
5. Predictions: both models on a grid, RMSE
6. Effect of the number of neighbors $m$


## 0. Setup

Build the C++ core and install the Julia binding (skip if already done).
Requires: `cmake`, a C++ compiler, Julia ≥ 1.10.

```shell
julia -e 'using Pkg; Pkg.develop(path="bindings/Julia/jlibkriging")'
```

In [ ]:
using jlibkriging
using Plots
using Random

println("jlibkriging loaded")

## 2. Branin function

In [ ]:
function branin(X::Matrix{Float64})
    x1 = X[:, 1] .* 15 .- 5
    x2 = X[:, 2] .* 15
    return (x2 .- 5 ./ (4π^2) .* x1.^2 .+ 5 ./ π .* x1 .- 6).^2 .+
           10 .* (1 - 1 / (8π)) .* cos.(x1) .+ 10
end

grid_x = collect(range(0, 1, length=50))
grid_pts = hcat(vec([x1 for x2 in grid_x, x1 in grid_x]),
                vec([x2 for x2 in grid_x, x1 in grid_x]))
z_true = branin(grid_pts)
contourf(grid_x, grid_x, reshape(z_true, 50, 50), title="True Branin")

## 3. Design of experiments (n = 1500)

In [ ]:
Random.seed!(123)
n = 1500
X = rand(n, 2)
y = branin(X)

## 4. Fit: `VLL(30)` vs exact `LL`

In [ ]:
t_vll = @elapsed k_vll = Kriging(y, X, "matern5_2"; objective="VLL(30)")
t_ll  = @elapsed k_ll  = Kriging(y, X, "matern5_2"; objective="LL")
println("fit VLL(30) : ", round(t_vll, digits=1), " s   theta = ", round.(theta(k_vll), digits=4))
println("fit LL      : ", round(t_ll, digits=1), " s   theta = ", round.(theta(k_ll), digits=4))
println("speedup     : ", round(t_ll / t_vll, digits=1), "x")

## 5. Predictions

In [ ]:
p_vll = predict(k_vll, grid_pts; return_stdev=false)
p_ll  = predict(k_ll, grid_pts; return_stdev=false)
display(contourf(grid_x, grid_x, reshape(p_vll.mean, 50, 50), title="mean — fit VLL(30)"))
println("RMSE vs true (VLL): ", sqrt(sum((p_vll.mean .- z_true).^2) / length(z_true)))
println("RMSE vs true (LL) : ", sqrt(sum((p_ll.mean .- z_true).^2) / length(z_true)))
println("max |VLL - LL|    : ", maximum(abs.(p_vll.mean .- p_ll.mean)))

## 6. Effect of the number of neighbors $m$

In [ ]:
for m in [5, 15, 30]
    t = @elapsed k = Kriging(y, X, "matern5_2"; objective="VLL($m)")
    p = predict(k, grid_pts; return_stdev=false)
    rmse = sqrt(sum((p.mean .- z_true).^2) / length(z_true))
    println("VLL(", lpad(m, 2), ") : fit ", round(t, digits=1), " s   RMSE = ", round(rmse, digits=4))
end

## Notes

- The screening effect behind Vecchia weakens in high dimension: recommended
  for $d \le 5$ (complementary to `NestedKriging`, which is dimension-robust).
- The final exact factorization at $\theta^*$ still costs $O(n^3)$ once. For
  very large $n$, the C++ API offers `predictVecchia` (local $O(q\,m^3)$
  prediction) and a "light" mode (`set_vecchia_exact_commit(false)`) that
  skips it entirely — bindings for both are planned.
- References: Vecchia (1988); Guinness (2018), *Technometrics*; Katzfuss &
  Guinness (2021), *Statistical Science*.
